In [ ]:
import json
from kafka import KafkaConsumer
from threading import Thread
import requests

# ----------------------------------------------------
# Telegram Config
TELEGRAM_TOKEN   = "8671227598:AAECm4bALvSoMdiCy_b7AgoEhZWMP0wEMEk"
TELEGRAM_CHAT_ID = "5570107538"

def send_telegram(message):
    url = f"https://api.telegram.org/bot{TELEGRAM_TOKEN}/sendMessage"
    requests.post(url, json={
        "chat_id": TELEGRAM_CHAT_ID,
        "text":    message,
        "parse_mode": "HTML"
    })

# ----------------------------
# Alert Rules
def check_system(data):
    alerts = []
    if data["cpu_pct"] > 85:
        alerts.append(f"🔴 CPU عالي: {data['cpu_pct']}%")
    if data["memory_pct"] > 85:
        alerts.append(f"🔴 Memory عالية: {data['memory_pct']}%")
    if data["battery_level_pct"] < 20:
        alerts.append(f"🔴 Battery منخفضة: {data['battery_level_pct']}%")
    return alerts

def check_radio(data):
    alerts = []
    if data["signal_dbm"] < -118:
        alerts.append(f" Signal ضعيف: {data['signal_dbm']} dBm")
    if data["drop_call_rate"] > 4:
        alerts.append(f"Drop Call عالي: {data['drop_call_rate']}%")
    return alerts

def check_environment(data):
    alerts = []
    if data["temperature_c"] > 45:
        alerts.append(f" Temperature عالية: {data['temperature_c']}°C")
    if data["humidity_pct"] > 90:
        alerts.append(f" Humidity عالية: {data['humidity_pct']}%")
    return alerts

def check_network(data):
    alerts = []
    if data["latency_ms"] > 350:
        alerts.append(f" Latency عالية: {data['latency_ms']}ms")
    if data["packet_loss"] > 0.12:
        alerts.append(f"Packet Loss عالي: {data['packet_loss']}")
    return alerts

# -------------------------------------------------
# Consumer لكل Topic

CHECKS = {
    "telecom.tower.system":      check_system,
    "telecom.tower.radio":       check_radio,
    "telecom.tower.environment": check_environment,
    "telecom.tower.network":     check_network,
}

def consume_topic(topic, check_fn):
    consumer = KafkaConsumer(
        topic,
        bootstrap_servers="localhost:9092",
        auto_offset_reset="latest",
        value_deserializer=lambda v: json.loads(v.decode("utf-8")),
        group_id=f"alert-service-{topic}",
    )
    print(f" Monitoring: {topic}")
    for msg in consumer:
        data   = msg.value
        alerts = check_fn(data)
        if alerts:
            message = (
                f" <b>Tower Alert!</b>\n"
                f" Tower: {data['tower_id']}\n"
                f" Region: {data['region']}\n"
                f" Time: {data['event_time']}\n\n"
                + "\n".join(alerts)
            )
            send_telegram(message)
            print(f"Alert sent for {data['tower_id']}")

# ------------------------------------------------------
# شغّل كل Consumer في Thread

threads = []
for topic, check_fn in CHECKS.items():
    t = Thread(target=consume_topic, args=(topic, check_fn), daemon=True)
    t.start()
    threads.append(t)

print(" Alert Microservice Started!")
print("-" * 50)

import time
try:
    while True:
        time.sleep(1)
except KeyboardInterrupt:
    print("Stopped!")